# Exercise Class 4: AD-AS Foundations, Oil Price Shocks, and the Taylor Rule

**Macroeconomics B -- Chapters 17-19**

By the end of this notebook you will be able to:
1. Build AD and SRAS functions from model primitives and solve for short-run equilibrium.
2. Plot the three-curve AD-AS diagram (AD, SRAS, LRAS) and mark equilibrium points.
3. Simulate how an oil price shock propagates over time under two Taylor rules.

**Table of contents**<a id='toc0_'></a>
- 1. [Parameters and model equations](#toc1_)
- 2. [Exercise P1: AD and SRAS toolkit](#toc2_)
- 3. [Exercise P2: Three-curve diagram](#toc3_)
- 4. [Exercise P3: Oil shock path simulation](#toc4_)
- 5. [[Optional] Exercise P4: Real FRED data](#toc5_)
- 6. [Summary](#toc6_)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

# Global plot settings (matches Programming for Economists style)
plt.rcParams.update({'axes.grid': True, 'grid.color': 'black',
                     'grid.alpha': 0.25, 'grid.linestyle': '--'})
plt.rcParams.update({'font.size': 14})

# Reproducibility
rng = np.random.default_rng(2026)

## 1. <a id='toc1_'></a>[Parameters and model equations](#toc0_)

*Corresponds to Exercise Sheet 4, all parts.*

The two core equations of the closed-economy AD-AS model are:

**AD curve** (derived from IS + Taylor rule):
$$y = \frac{\bar A - ar^n}{1+ab} - \frac{ah}{1+ab}(\pi - \pi^T)$$

**SRAS curve** (derived from Phillips curve + Okun):
$$\pi = \pi^e + \gamma y + s$$

where $s > 0$ is an adverse cost-push shock (e.g. an oil price spike).

**LRAS:** In the long run $\pi = \pi^e$, which implies $y = 0$ (potential output).
The LRAS is a vertical line at $y = 0$.

All parameters are collected in a single dict. Edit the dict to explore.

In [ ]:
# -------------------------------------------------------
# Baseline parameters -- edit these to explore
# -------------------------------------------------------
par = {
    'A_bar':   2.0,   # autonomous demand (bar A)
    'a':       1.0,   # IS interest sensitivity
    'r_n':     2.0,   # natural real interest rate
    'pi_star': 2.0,   # inflation target
    'h':       0.75,  # Taylor inflation response
    'b':       0.5,   # Taylor output-gap response
    'gamma':   0.5,   # AS slope (Okun x Phillips)
    'pi_e':    2.0,   # expected inflation
    's':       0.0,   # supply shock (s > 0 = adverse)
}

# Confirm the normalisation: A_bar - a*r_n should equal 0
print(f"A_bar - a*r_n = {par['A_bar'] - par['a']*par['r_n']:.3f}")
print("(When this equals 0, the economy is at potential when pi = pi_star.)")

## 2. <a id='toc2_'></a>[Exercise P1: AD and SRAS toolkit](#toc0_)

*Corresponds to Exercise Sheet 4, Part VI, Exercise P1.*

Complete the three function skeletons below.
The docstrings describe exactly what each function should do.

In [ ]:
def ad_output(pi, p):
    """ Return the output gap implied by the AD curve at inflation pi.

    Args:
        pi  (float or np.ndarray): inflation rate
        p   (dict): model parameters

    Returns:
        y   (float or np.ndarray): output gap

    Formula:
        y = (A_bar - a*r_n) / (1 + a*b) - (a*h) / (1 + a*b) * (pi - pi_star)
    """
    # TODO: unpack the parameters you need
    # TODO: compute and return the AD output gap
    raise NotImplementedError


def sras_inflation(y, p):
    """ Return the inflation rate implied by the SRAS curve at output gap y.

    Args:
        y   (float or np.ndarray): output gap
        p   (dict): model parameters, must contain 'pi_e' and 's'

    Returns:
        pi  (float or np.ndarray): inflation

    Formula:
        pi = pi_e + gamma * y + s
    """
    # TODO: unpack and compute
    raise NotImplementedError


def solve_sras_ad(p):
    """ Solve the SRAS-AD system analytically.

    Substitute AD (y as function of pi) into SRAS (pi as function of y)
    and solve the resulting single equation for pi, then back out y.

    Args:
        p   (dict): model parameters

    Returns:
        dict with keys 'y_star' and 'pi_eq'

    Hint: the AD equation gives y = intercept + slope * (pi - pi_star).
    Substituting into SRAS gives a single linear equation in pi.
    """
    # TODO: compute the AD slope and intercept
    # ad_slope = ...
    # ad_intercept = ...

    # TODO: substitute AD into SRAS and solve for pi_star
    # pi_eq = ...

    # TODO: back out y_star from AD
    # y_eq = ...

    raise NotImplementedError

In [ ]:
# Verification: run this cell after completing the functions above.
# Expected output:
#   s = 0.0  -->  y* =  0.0, pi* = 2.0
#   s = 1.5  -->  y* = -0.6, pi* = 3.2

for s_val in [0.0, 1.5]:
    par_check = par.copy()
    par_check['s'] = s_val
    eq = solve_sras_ad(par_check)
    print(f"s = {s_val:.1f}  -->  y* = {eq['y_star']:6.3f}, pi* = {eq['pi_eq']:.3f}")

## 3. <a id='toc3_'></a>[Exercise P2: Three-curve diagram](#toc0_)

*Corresponds to Exercise Sheet 4, Part VI, Exercise P2.*

Plot AD, baseline SRAS ($s=0$), oil-shock SRAS ($s=1.5$), and LRAS on the same axes.
Mark the three equilibria $E_0$, $E_1$, and $E_1^H$ (hawkish AD).

In [ ]:
# -------------------------------------------------------
# Grid for plotting
# -------------------------------------------------------
y_grid = np.linspace(-2.5, 1.5, 300)
pi_grid = np.linspace(0.5, 5.5, 300)

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
# -------------------------------------------------------
# Three-curve AD-AS diagram
# -------------------------------------------------------
par_base   = par.copy()           # s = 0, h = 0.75 (baseline)
par_shock  = par.copy()           # s = 1.5, h = 0.75
par_shock['s'] = 1.5
par_hawk   = par.copy()           # s = 1.5, h = 1.5 (hawkish)
par_hawk['s'] = 1.5
par_hawk['h'] = 1.5

fig = plt.figure(figsize=(8.5, 5.5))
ax = fig.add_subplot(1, 1, 1)

# --- AD curves (plot as pi on y-axis, y on x-axis) ---
# AD gives y as a function of pi; we plot pi_grid on vertical axis
# and compute y = ad_output(pi_grid, par) on horizontal axis.

# TODO: plot baseline AD (h=0.75)
# ax.plot(ad_output(pi_grid, par_base), pi_grid, ...)

# TODO: plot hawkish AD (h=1.5)
# ax.plot(ad_output(pi_grid, par_hawk), pi_grid, ...)

# --- SRAS curves (plot as pi = sras_inflation(y, par) against y_grid) ---
# TODO: plot baseline SRAS (s=0)
# ax.plot(y_grid, sras_inflation(y_grid, par_base), ...)

# TODO: plot oil-shock SRAS (s=1.5)
# ax.plot(y_grid, sras_inflation(y_grid, par_shock), ...)

# --- LRAS (vertical line at y = 0) ---
# TODO: add vertical LRAS line
# ax.axvline(0, ...)

# --- Equilibria (scatter points) ---
# TODO: solve and mark E0 (s=0, baseline AD)
# TODO: solve and mark E1 (s=1.5, baseline AD, h=0.75)
# TODO: solve and mark E1_H (s=1.5, hawkish AD, h=1.5)

ax.set_xlim(-2.5, 1.5)
ax.set_ylim(0.5, 5.5)
ax.set_xlabel('Output gap, $y$')
ax.set_ylabel(r'Inflation, $\pi$')
ax.set_title('AD-AS diagram: energy shock and policy response', pad=10)
ax.axhline(par['pi_star'], color='black', lw=0.8, ls=':', alpha=0.5)
ax.legend(loc='upper right', fontsize=11)

fig.tight_layout()

**What is done:** *[complete this after running the figure]*

**Why it is useful:** *[connect to the policy dilemma from Part V of the exercise sheet]*

**How to interpret the result:** *[describe what $E_1$ and $E_1^H$ tell you about the trade-off]*

## 4. <a id='toc4_'></a>[Exercise P3: Oil shock path simulation](#toc0_)

*Corresponds to Exercise Sheet 4, Part VI, Exercise P3.*

Simulate the 2022-style energy shock over four periods.
The shock fades: $s = [1.5,\ 1.0,\ 0.5,\ 0.0]$.

Expectations are **static**: $\pi^e_t = \pi_{t-1}$.
In period 1 (before the shock hits), $\pi^e_1 = \pi^T = 2$.

In [ ]:
def simulate_oil_shock(p, h_override, s_path):
    """ Simulate the AS-AD model with a time-varying supply shock.

    Uses static expectations: pi_e_{t} = pi_{t-1}.
    Initial expected inflation equals pi_star.

    Args:
        p          (dict):        baseline parameter dict
        h_override (float):       Taylor inflation response to use (overrides p['h'])
        s_path     (list/array):  supply shock in each period

    Returns:
        pd.DataFrame with columns: period, s, pi_e, y, pi
    """
    # a. copy parameters and set the override
    par_sim = p.copy()
    par_sim['h'] = h_override

    T = len(s_path)

    # b. preallocate
    periods = np.empty(T, dtype=int)
    s_arr   = np.empty(T)
    pi_e_arr = np.empty(T)
    y_arr   = np.empty(T)
    pi_arr  = np.empty(T)

    # c. initial expected inflation = pi_star
    pi_prev = par_sim['pi_star']

    # d. iterate forward
    for t in range(T):
        periods[t]  = t + 1
        s_arr[t]    = s_path[t]
        pi_e_arr[t] = pi_prev

        # TODO: set current pi_e and s in par_sim
        # par_sim['pi_e'] = ...
        # par_sim['s']    = ...

        # TODO: solve for equilibrium y and pi in this period
        # eq = solve_sras_ad(par_sim)
        # y_arr[t]  = ...
        # pi_arr[t] = ...

        # TODO: update lagged inflation for next period
        # pi_prev = ...

        raise NotImplementedError   # remove this line when done

    return pd.DataFrame({
        'period': periods,
        's':      s_arr,
        'pi_e':   pi_e_arr,
        'y':      y_arr,
        'pi':     pi_arr,
    })

In [ ]:
# -------------------------------------------------------
# Run the simulation for both Taylor rules
# -------------------------------------------------------
s_path = [1.5, 1.0, 0.5, 0.0]

# TODO: run for h = 0.75 (baseline)
# df_base = simulate_oil_shock(par, h_override=0.75, s_path=s_path)

# TODO: run for h = 1.5 (hawkish)
# df_hawk = simulate_oil_shock(par, h_override=1.50, s_path=s_path)

# Preview baseline results
# df_base

In [ ]:
# -------------------------------------------------------
# Two-panel figure: output gap and inflation over time
# -------------------------------------------------------
fig = plt.figure(figsize=(11.5, 4.5))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2)

# TODO: plot output gap for both rules on ax1
# ax1.plot(df_base['period'], df_base['y'], ...)
# ax1.plot(df_hawk['period'], df_hawk['y'], ...)

# TODO: plot inflation for both rules on ax2
# ax2.plot(df_base['period'], df_base['pi'], ...)
# ax2.plot(df_hawk['period'], df_hawk['pi'], ...)

ax1.axhline(0, color='black', lw=1, ls='--')
ax2.axhline(par['pi_star'], color='black', lw=1, ls='--', label=r'$\pi^T$')

ax1.set_title('Output gap', pad=10)
ax1.set_xlabel('Period')
ax1.set_ylabel('Output gap, $y$')

ax2.set_title('Inflation', pad=10)
ax2.set_xlabel('Period')
ax2.set_ylabel(r'Inflation, $\pi$')

for ax in [ax1, ax2]:
    ax.legend(loc='best')

fig.tight_layout()

**Interpretation:** *Write two sentences below.*

*Which rule brings inflation back to target faster, and what is the output cost?*

*Your answer here.*

## 5. <a id='toc5_'></a>[[Optional] Exercise P4: Real FRED data](#toc0_)

*Corresponds to Exercise Sheet 4, Part VI, Exercise P4.*

Download EU natural gas prices and euro-area HICP from FRED.
Compare the model simulation to the actual inflation episode.

In [ ]:
# -------------------------------------------------------
# FRED API setup
# Get your free key at: https://fredaccount.stlouisfed.org/apikey
# -------------------------------------------------------
FRED_API_KEY = 'YOUR_KEY_HERE'   # <-- replace with your own key

FRED_BASE = 'https://api.stlouisfed.org/fred/series/observations'

def fetch_fred(series_id, api_key, start='2020-01-01'):
    """ Download a FRED series as a pandas Series with DatetimeIndex.

    Args:
        series_id (str): FRED series identifier
        api_key   (str): your FRED API key
        start     (str): start date in 'YYYY-MM-DD' format

    Returns:
        pd.Series with DatetimeIndex
    """
    # a. build request
    params = {
        'series_id': series_id,
        'api_key': api_key,
        'file_type': 'json',
        'observation_start': start,
    }

    # b. fetch
    r = requests.get(FRED_BASE, params=params, timeout=30)
    r.raise_for_status()

    # c. parse
    obs    = r.json()['observations']
    dates  = pd.to_datetime([o['date']  for o in obs])
    values = pd.to_numeric([o['value'] for o in obs], errors='coerce')
    s = pd.Series(values.values, index=dates, name=series_id)
    print(f'Fetched {series_id}: {s.index[0].date()} to {s.index[-1].date()}, '
          f'{s.notna().sum()} non-null observations.')
    return s

In [ ]:
# TODO: download European natural gas prices and euro-area HICP
# gas     = fetch_fred('PNGASEUUSDM', FRED_API_KEY)
# hicp_ea = fetch_fred('CP0000EZ19M086NEST', FRED_API_KEY)

# TODO: compute year-on-year change in HICP
# hicp_yoy = hicp_ea.pct_change(12) * 100

# TODO: compute year-on-year change in gas prices
# gas_yoy = gas.pct_change(12) * 100

In [ ]:
# TODO: dual-axis figure -- HICP inflation (left axis) and gas price change (right axis)
# Use fig = plt.figure() / ax = fig.add_subplot() pattern.
# Add ax2 = ax1.twinx() for the gas price series.
# Shade the period 2022-02 to 2022-12 with ax1.axvspan().

# write your code here

## 6. <a id='toc6_'></a>[Summary](#toc0_)

| Equation | Meaning |
|---|---|
| $i = r^n + \pi + h(\pi - \pi^T) + by$ | **Taylor rule**: nominal rate responds to inflation and output gap |
| $y = -\frac{ah}{1+ab}(\pi - \pi^T)$ | **AD curve** (with $\bar A = ar^n$): higher inflation lowers demand via policy tightening |
| $\pi = \pi^e + \gamma y + s$ | **SRAS curve**: inflation rises with output and supply shocks |
| $y = 0$ | **LRAS**: output at potential in the long run (when $\pi = \pi^e$, $s = 0$) |
| $y^{eq} = -0.6,\ \pi^{eq} = 3.2$ | **Short-run equilibrium** after $s = 1.5$ oil shock (baseline $h = 0.75$) |
| $y^{eq} = -1.0,\ \pi^{eq} = 3.0$ | **Short-run equilibrium** after $s = 1.5$ oil shock (hawkish $h = 1.50$) |

**Socrative room:** MACROECONOMICSB